In [ ]:
import pandas as pd
import numpy as np

def transform_skewed_features(df):
    """
    Applies logarithmic compression to right-skewed features and 
    a power stretch to left-skewed features.
    """
    print("--- Transforming Skewed Features ---")
    df_out = df.copy()
    
    # 1. Fixing Right-Skew (Logarithmic compression)
    right_skewed_features = ["Wind speed", "Solar Radiation", "Rainfall", "Snowfall"]
    
    # Only process columns that exist in the dataframe
    existing_right_cols = [col for col in right_skewed_features if col in df_out.columns]
    if existing_right_cols:
        log1p_cols = [f"{col}_log1p" for col in existing_right_cols]
        df_out[log1p_cols] = np.log1p(df_out[existing_right_cols])

    # 2. Fixing Left-Skew (Power stretching)
    if "Visibility" in df_out.columns:
        df_out["Visibility_ref_log"] = df_out["Visibility"] ** 3
        
    print("Transformations complete.")
    return df_out


def categorize_weather_features(df):
    """
    Categorizes heavily skewed weather features with point-mass "walls" 
    into discrete ordinal bins.
    """
    print("--- Categorizing Skewed Weather Features ---")
    df_out = df.copy()
    
    # 1. Visibility (Left-skewed wall at 2000)
    if 'Visibility' in df_out.columns:
        vis_bins = [-1, 500, 1500, np.inf]
        df_out['Visibility_Cat'] = pd.cut(df_out['Visibility'], bins=vis_bins, labels=[0, 1, 2]).astype("Int64")

    # 2. Solar Radiation (Right-skewed wall at 0 during night/clouds)
    if 'Solar Radiation' in df_out.columns:
        solar_bins = [-0.1, 0.1, 1.5, np.inf]
        df_out['Solar_Radiation_Cat'] = pd.cut(df_out['Solar Radiation'], bins=solar_bins, labels=[0, 1, 2]).astype("Int64")

    # 3. Rainfall (Right-skewed wall at 0)
    if 'Rainfall' in df_out.columns:
        rain_bins = [-0.1, 0.1, 2.0, np.inf]
        df_out['Rainfall_Cat'] = pd.cut(df_out['Rainfall'], bins=rain_bins, labels=[0, 1, 2]).astype("Int64")

    # 4. Snowfall (Right-skewed wall at 0)
    if 'Snowfall' in df_out.columns:
        snow_bins = [-0.1, 0.1, 1.0, np.inf]
        df_out['Snowfall_Cat'] = pd.cut(df_out['Snowfall'], bins=snow_bins, labels=[0, 1, 2]).astype("Int64")

    print("Categorization complete.")
    return df_out